# 2. Hypothesis Testing

Tests whether the observed difference in conversion rates between the ad and psa groups is statistically significant.

## Load Data

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

path = "ab_marketing/marketing_ab.csv"
df = pd.read_csv(path)

try:
    df.drop(columns=["Unnamed: 0"], inplace=True)
    df.rename(columns={
        'user id': 'user_id',
        'test group': 'test_group',
        'total ads': 'total_ads',
        'most ads day': 'most_ads_day',
        'most ads hour': 'most_ads_hour'
    }, inplace=True)
except KeyError:
    print('Cell already run.')

## Hypothesis Setup

- **H₀**: The conversion rate of the ad group equals that of the psa group (p_ad = p_psa)
- **H₁**: The conversion rates differ (p_ad ≠ p_psa)
- **Alpha**: 0.05, two-tailed test

## Group Metrics

In [ ]:
ad = df[df['test_group'] == 'ad']
psa = df[df['test_group'] == 'psa']

n_ad, n_psa = len(ad), len(psa)
conv_ad, conv_psa = ad['converted'].sum(), psa['converted'].sum()
p_ad, p_psa = conv_ad / n_ad, conv_psa / n_psa

print(f'ad: n={n_ad:,} | conversions={conv_ad:,} | rate={p_ad:.4f}')
print(f'psa: n={n_psa:,} | conversions={conv_psa:,} | rate={p_psa:.4f}')

## Z-Test

In [ ]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

z_stat, p_value = proportions_ztest(
    np.array([conv_ad, conv_psa]),
    np.array([n_ad, n_psa]),
    alternative='two-sided'
)

alpha = 0.05
print(f'Z-statistic : {z_stat:.4f}')
print(f'P-value : {p_value:.2e}')
print(f'Reject H₀ : {p_value < alpha}')

## Confidence Interval

In [ ]:
p_diff = p_ad - p_psa
se = np.sqrt(p_ad*(1-p_ad)/n_ad + p_psa*(1-p_psa)/n_psa)
ci_lower = p_diff - 1.96 * se
ci_upper = p_diff + 1.96 * se

print(f'Absolute difference : {p_diff*100:.4f} pp')
print(f'Relative lift : {(p_diff/p_psa)*100:.1f}%')
print(f'95% CI : ({ci_lower*100:.4f}, {ci_upper*100:.4f}) pp')

## Effect Size

In [ ]:
from statsmodels.stats.proportion import proportion_effectsize

h = proportion_effectsize(p_ad, p_psa)

thresholds = [(0.2, 'negligible'), (0.5, 'small'), (0.8, 'medium'), (float('inf'), 'large')]
magnitude = next(label for limit, label in thresholds if abs(h) < limit)

print(f"Cohen's h : {h:.4f} ({magnitude})")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Conversion rates with individual CIs
groups = ['ad', 'psa']
rates  = [p_ad, p_psa]
errors = [1.96 * np.sqrt(p*(1-p)/n) for p, n in zip(rates, [n_ad, n_psa])]
colors = ['#4C72B0', '#DD8452']

bars = axes[0].bar(groups, [r*100 for r in rates], color=colors, alpha=0.85, width=0.4)
axes[0].errorbar(groups, [r*100 for r in rates], yerr=[e*100 for e in errors],
                fmt='none', color='black', capsize=6, linewidth=1.5) 
for bar, rate in zip(bars, rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, rate*100 + 0.12,
                f'{rate*100:.2f}%', ha='center', fontsize=10)
axes[0].set_title('Conversion Rate by Group (95% CI)', fontweight='bold')
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].set_ylim(0, max(rates)*100*1.3)

# CI on the difference
axes[1].barh(['ad − psa'], [p_diff*100], color='#4C72B0', alpha=0.8, height=0.3)
axes[1].errorbar([p_diff*100], ['ad − psa'],
                xerr=[[(p_diff-ci_lower)*100], [(ci_upper-p_diff)*100]],
                fmt='none', color='black', capsize=6, linewidth=1.5)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Difference in Conversion Rate (pp)')
axes[1].set_title('95% CI on the Difference', fontweight='bold')

plt.suptitle('Hypothesis Test Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Results Summary

In [ ]:
summary = pd.DataFrame({
    'group' : ['ad', 'psa'],
    'n' : [n_ad, n_psa],
    'conversions' : [conv_ad, conv_psa],
    'conversion_rate' : [round(p_ad, 4), round(p_psa, 4)],
}).set_index('group')

display(summary)
print(f"\nZ-statistic : {z_stat:.4f}")
print(f"P-value : {p_value:.2e}")
print(f"95% CI : ({ci_lower*100:.4f}, {ci_upper*100:.4f}) pp")
print(f"Cohen's h : {h:.4f} ({magnitude})")